In [ ]:
import torch
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader



import sys, os
sys.path.append(os.path.abspath(os.path.join(os.getcwd(),'..' ,'..')))
from Data.class_dataset import MRIDataset
from Model.SFCN import SFCN

# --- Cargar datos de test ---

df=pd.read_csv("test_nico.csv")

test_imgs = df["Path"].tolist()
test_ages = df["Age"].tolist()
test_dataset = MRIDataset(test_imgs, test_ages,mode='val')
test_loader = DataLoader(test_dataset, batch_size=8, shuffle=False)

# --- Cargar modelo ---
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = SFCN()
state_dict = torch.load("../../Training/Trained_models/model_nico.pth", map_location=device)
# Si las claves tienen 'module.' al inicio, elimínalo
if any(k.startswith('module.') for k in state_dict.keys()):
    from collections import OrderedDict
    new_state_dict = OrderedDict()
    for k, v in state_dict.items():
        new_key = k.replace('module.', '', 1)
        new_state_dict[new_key] = v
    state_dict = new_state_dict
model.load_state_dict(state_dict)
model.to(device)
model.eval()



/home/german/.local/lib/python3.10/site-packages/matplotlib/projections/__init__.py:63: UserWarning: Unable to import Axes3D. This may be due to multiple versions of Matplotlib being installed (e.g. as a system package and as a pip package). As a result, the 3D projection is not available.
  warnings.warn("Unable to import Axes3D. This may be due to multiple versions of "


SFCN(
  (block1): ConvBnRelu(
    (block): Sequential(
      (0): Conv3d(1, 32, kernel_size=(3, 3, 3), stride=(1, 1, 1), padding=(1, 1, 1), bias=False)
      (1): BatchNorm3d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): ReLU(inplace=True)
      (3): MaxPool3d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    )
  )
  (block2): ConvBnRelu(
    (block): Sequential(
      (0): Conv3d(32, 64, kernel_size=(3, 3, 3), stride=(1, 1, 1), padding=(1, 1, 1), bias=False)
      (1): BatchNorm3d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): ReLU(inplace=True)
      (3): MaxPool3d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    )
  )
  (block3): ConvBnRelu(
    (block): Sequential(
      (0): Conv3d(64, 128, kernel_size=(3, 3, 3), stride=(1, 1, 1), padding=(1, 1, 1), bias=False)
      (1): BatchNorm3d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): ReLU(inplace=Tru

In [ ]:
# --- Evaluar ---
all_preds = []
all_ages = []
with torch.no_grad():
    for imgs, ages, _, _  in test_loader:
        imgs = imgs.to(device)
        preds = model(imgs)
        all_preds.append(preds.cpu())
        all_ages.append(ages.unsqueeze(1).cpu())
all_preds = torch.cat(all_preds).numpy().flatten()
all_ages = torch.cat(all_ages).numpy().flatten()
all_paths = df["Path"].tolist()

mae = mean_absolute_error(all_ages, all_preds)
r2 = r2_score(all_ages, all_preds)
print(f"Test MAE: {mae:.2f}")
print(f"Test R2: {r2:.2f}")

ValueError: too many values to unpack (expected 2)

In [ ]:
#crear un df con las edades reales, predichas y el ID
results_df = pd.DataFrame({
    "ID": all_paths,
    "Age": all_ages,
    "Prediction": all_preds,
    "Error": all_preds - all_ages, 
    'Absolute Error': np.abs(all_preds - all_ages)
})
results_df['ID']=results_df['ID'].replace('/data/lautaro/quasiraw_ext/papers/', '', regex=True)
results_df['ID']=results_df['ID'].replace('.nii.gz', '', regex=True)
results_df.sort_values(by='Absolute Error', ascending=True, inplace=True)
results_df.to_csv("test_papers_predictions_11.csv", index=False)

In [ ]:
results_df